# 01. Auditoría inicial y criterios de calidad

Este cuaderno examina los datos GMN de diciembre de 2019 para revisar su estructura, detectar problemas de calidad y estudiar los criterios que después se aplican al conjunto multianual.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 1. Carga y estructura de los datos


In [ ]:
# Definir la raíz del proyecto y la carpeta de datos originales

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"

PROJECT_ROOT


In [ ]:
# Construir la ruta del archivo de 2019 y comprobar que está disponible

raw_file = RAW_DIR / "traj_summary_monthly_201912.txt"

if not raw_file.exists():
    raise FileNotFoundError(
        "No se encontró data/raw/traj_summary_monthly_201912.txt. "
        "Descarga los datos indicados en el README antes de ejecutar los notebooks."
    )

print("Archivo encontrado: ", raw_file.exists())
print("Ruta: ", raw_file)
print("Tamaño en MB: ", raw_file.stat().st_size / 1_000_000)


In [ ]:
# Mostrar las primeras diez líneas sin modificar ni cargar todavía toda la base

with raw_file.open(
    "r",
    encoding = "utf-8",
    errors = "replace"
) as file:
    for _ in range(10):
        print(file.readline().rstrip())


In [ ]:
# Cargar el archivo GMN como DataFrame conservando sus dos filas de encabezado

df_raw = pd.read_csv(
    raw_file,
    engine = "python",
    sep = ";",
    skipinitialspace = True,
    skiprows = [0, 5, 6],
    header = [0, 1],
    na_values = ["nan", "None"],
)

print("Número de filas: ", df_raw.shape[0])
print("Número de columnas: ", df_raw.shape[1])

display(df_raw.head(3))


In [ ]:
# Mostrar los primeros encabezados

print("Tipos del encabezado: ", type(df_raw.columns))
print("Número de niveles", df_raw.columns.nlevels)

display(
    pd.DataFrame(
        df_raw.columns[:15].tolist(),
        columns = ["Nivel 1: nombre", "Nivel 2: unidad"],
    )
)


## 2. Limpieza estructural


In [ ]:
# Unir los dos niveles del encabezado en un solo nombre por columna

def clean_header_text(text):
    cleaned_text = str(text).replace("#", "")
    cleaned_text = " " .join(cleaned_text.split())

    return cleaned_text


flattened_columns = []

for main_name, secondary_name in df_raw.columns:
    main_name = clean_header_text(main_name)
    secondary_name = clean_header_text(secondary_name)

    if secondary_name.startswith("Unnamed"):
        final_name = main_name
    else:
        final_name = f"{main_name} ({secondary_name})"

    flattened_columns.append(final_name)


df_raw.columns = flattened_columns

for column_name in df_raw.columns[:15]:
    print(column_name)


In [ ]:
# Renombrar cada columna de incertidumbre usando el nombre de la variable anterior

renamed_columns = list(df_raw.columns)

for column_position, column_name in enumerate(renamed_columns):
    if column_name.startswith("+/-"):
        previous_column = renamed_columns[column_position - 1]
        parameter_name = previous_column.split(" ")[0]
        
        renamed_columns[column_position] = (
            f"{parameter_name}_sigma"
        )
        
df_raw.columns = renamed_columns

print("¿Todos los nombres son únicos?: ", df_raw.columns.is_unique)

for column_name in df_raw.columns:
    if column_name.endswith("_sigma"):
        print(column_name)


In [ ]:
# Comprobar dimensiones, tipos generales y primeras columnas del DataFrame cargado

print("Filas: ", df_raw.shape[0])
print("Columnas: ", df_raw.shape[1])
print("Nombres únicos: ", df_raw.columns.is_unique)

print("\n Información general: ")
df_raw.info(verbose = False)

print("\n Primeras tres filas y primeras doce columnas: ")

display(df_raw.iloc[:3, :12])


In [ ]:
# Crear una copia del trabajo y registrar el archivo y año de procedencia

df = df_raw.copy()

df["source_file"] = raw_file.name
df["source_year"] = 2019

print("Dimensiones del original: ", df_raw.shape)
print("Dimensiones de la copia: ", df.shape)

display(
    df[
        [
            "Unique trajectory (identifier)",
            "source_file",
            "source_year",
        ]
    ].head(3)
)


In [ ]:
# Normalizar los textos y registrar si había campos vacíos no reconocidos como faltantes

text_columns = [
    "Unique trajectory (identifier)",
    "IAU (code)",
    "Participating (stations)",
]

missing_before = df_raw[text_columns].isna().sum()

for column in text_columns:
    df[column] = (df_raw[column].astype("string").str.strip().replace("", pd.NA))

text_report = pd.DataFrame(
    {
        "faltantes_antes": missing_before,
        "faltantes_después": df[text_columns].isna().sum(),
    }
)

display(text_report)
display(df[text_columns].head(3))


In [ ]:
# Convertir las fechas a datetime en UTx y comprobar su rango y valores faltantes

date_column = "Beginning (UTC Time)"

date_text = (df_raw[date_column].astype("string").str.strip().replace("", pd.NA))

df[date_column] = pd.to_datetime(date_text, format = "%Y-%m-%d %H:%M:%S.%f", errors = "raise", utc = True)

print("Tipos después de convertir: ", df[date_column].dtype)
print("Fechas faltantes antes: ", date_text.isna().sum())
print("Fechas faltantes después: ", df[date_column].isna().sum())
print("Fecha más temprana: ", df[date_column].min())
print("Fecha más tardía: ", df[date_column].max())

display(
    pd.DataFrame(
        {
            "texto_original ": df_raw[date_column].head(3),
            "fecha_interpretada ": df[date_column].head(3),
        }
    )
)


In [ ]:
# Revisar los tipos principales y contar los valores de las columnas booleanas

columns_to_check = [
    "Unique trajectory (identifier)",
    "Beginning (UTC Time)",
    "IAU (No)",
    "IAU (code)",
    "Num (stat)",
    "Beg in (FOV)",
    "End in (FOV)",
    "Participating (stations)",
]

type_report = df[columns_to_check].dtypes.to_frame(name = "tipo")

display(type_report)

boolean_columns = [
    "Beg in (FOV)",
    "End in (FOV)",
]

for column in boolean_columns:
    print("\n Columna:", column)
    display(df[column].value_counts(dropna = False))
    
print("\n Filas originales: ", len(df_raw))
print("Filas actuales: ", len(df))


## 3. Integridad de las observaciones


In [ ]:
# Resumir la cantidad y el porcentaje de variables faltantes por columna

missing_report = pd.DataFrame(
    {
        "disponibles": df.notna().sum(),
        "faltantes": df.isna().sum(),
        "porcentaje_faltante": df.isna().mean() * 100,
    }
)

missing_report = missing_report.sort_values(
    "porcentaje_faltante",
    ascending = False,
)

print("Número de filas analizadas: ", len(df))
print("Columnas con al menos un faltante: ", (missing_report["faltantes"] > 0).sum())
display(
    missing_report.loc[
        missing_report["faltantes"] > 0
    ].round(2)
)


In [ ]:
# Revisar la disponiblidad de las variables importantes para el análisis
analysis_columns = [
    "Unique trajectory (identifier)",
    "Beginning (UTC Time)",
    "IAU (code)",
    "Sol lon (deg)",
    "RAgeo (deg)",
    "RAgeo_sigma",
    "DECgeo (deg)",
    "DECgeo_sigma",
    "LAMgeo (deg)",
    "BETgeo (deg)",
    "Vgeo (km/s)",
    "Vgeo_sigma",
    "Peak (AbsMag)",
    "Qc (deg)",
    "MedianFitErr (arcsec)",
    "Num (stat)",
    "Beg in (FOV)",
    "End in (FOV)",
]

analysis_missing_report = missing_report.loc[analysis_columns].copy()

display(analysis_missing_report.round(2))


In [ ]:
# Detectar filas duplicadas e identificadores de trayectoria repetidos sin eliminar observaciones

id_column = "Unique trajectory (identifier)"

duplicate_id_mask = df.duplicated(subset = [id_column], keep = False)

duplicate_summary = pd.DataFrame(
    {
        "medida": [
            "Filas totales",
            "Identificadores únicos",
            "Filas con un ID repetido",
            "Identificadores que aparecen más de una vez",
            "Filas adicionales por repetición de ID",
            "Filas completamente duplicadas adicionales"
        ],
        "valor": [
            len(df),
            df[id_column].nunique(dropna = False),
            duplicate_id_mask.sum(),
            df.loc[duplicate_id_mask, id_column].nunique(),
            df.duplicated(subset = [id_column], keep = "first").sum(),
            df.duplicated(keep = "first").sum(),
        ],
    }
)

display(duplicate_summary)

if duplicate_id_mask.any():
    duplicate_review_columns = [
        id_column,
        "Beginning (UTC Time)",
        "IAU (No)",
        "IAU (code)",
        "Sol lon (deg)",
        "RAgeo (deg)",
        "DECgeo (deg)",
        "Vgeo (km/s)",
        "Peak (AbsMag)",
        "source_file",
    ]
    
    duplicate_review = (
        df.loc[
            duplicate_id_mask,
            duplicate_review_columns,
        ]
        .sort_values(id_column)
    )
    
    print("Primeros identificadores repetidos: ")
    display(duplicate_review.head(20))
    
else:
    print("No se encontraron identificadores repetidos.")


In [ ]:
# Comprobar que las variables principales estén dentro de rangos matemáticos y físicos válidos

reglas_validez = {
    "Longitud solar fuera de [0, 360)": (
        df["Sol lon (deg)"].ge(0)
        & df["Sol lon (deg)"].lt(360)
    ),
    
    "Ascensión recta fuera de [0, 360)": (
      df["RAgeo (deg)"].ge(0)
      & df["RAgeo (deg)"].lt(360)
    ),
    
    "Declinación fuera de [-90, 90]": (
        df["DECgeo (deg)"].between(-90,90)
    ),
    
    "Longitud eclíptica fuera de [0, 360)": (
        df["LAMgeo (deg)"].ge(0)
        & df["LAMgeo (deg)"].lt(360)
    ),
    
    "Latitud eclíptica fuera de [-90, 90]": (
        df["BETgeo (deg)"].between(-90, 90)
    ),
    
    "Velocidad geocéntrica no positiva": (
        df["Vgeo (km/s)"].gt(0)
    ),
    
    "Incertidumbre de RA negativa": (
        df["RAgeo_sigma"].ge(0)
    ),
    
    "Incertidumbre de DEC negativa": (
        df["DECgeo_sigma"].ge(0)
    ),
    
    "Incertidumbre de velocidad negativa": (
        df["Vgeo_sigma"].ge(0)
    ),
    
    "Ángulo de convergencia fuera de [0, 180]": (
        df["Qc (deg)"].between(0, 180)
    ),
    
    "Error de ajuste negativo": (
        df["MedianFitErr (arcsec)"].ge(0)
    ),
    
    "Menos de dos estaciones": (
        df["Num (stat)"].ge(2)
    )
}

resultados_rangos = []

for descripcion, mascara_valida in reglas_validez.items():
    mascara_invalida = ~mascara_valida.fillna(False)
    cantidad_invalida = mascara_invalida.sum()
    
    resultados_rangos.append(
        {
            "comprobación": descripcion,
            "filas_invalidas": cantidad_invalida,
            "porcentaje_invalido": round(
                100 * cantidad_invalida / len(df),
                4,
            ),
        }
    )
    
auditoria_rangos = pd.DataFrame(resultados_rangos)

display(auditoria_rangos)


## 4. Diagnóstico de calidad


In [ ]:
# Resumir la distribución de las variables físicas y de calidad antes de definir filtros

columnas_diagnostico = [
    "RAgeo_sigma",
    "DECgeo_sigma",
    "Vgeo_sigma",
    "Qc (deg)",
    "MedianFitErr (arcsec)",
    "Num (stat)",
    "Vgeo (km/s)",
    "Peak (AbsMag)",
]

resumen_diagnostico = (
    df[columnas_diagnostico]
    .describe(
        percentiles = [
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
    .T
    .rename(
        columns = {
            "count": "cantidad",
            "mean": "media",
            "std": "desviacion_estandar",
            "min": "minimo",
            "1%": "percentil_1",
            "5%": "percentil_5",
            "25%": "percentil_25",
            "50%": "mediana",
            "75%": "percentil_75",
            "95%": "percentil_95",
            "99%": "percentil_99",
            "max": "maximo",
        }
    )
    .round(4)
)

display(resumen_diagnostico)


In [ ]:
columnas_incertidumbre = [
    "RAgeo_sigma",
    "DECgeo_sigma",
    "Vgeo_sigma",
    "MedianFitErr (arcsec)",
]

resumen_minimos = pd.DataFrame(
    {
        "variable": columnas_incertidumbre,
        "minimo_exacto": [
            df[columna].min()
            for columna in columnas_incertidumbre
        ],
        "valores_iguales_a_cero": [
            df[columna].eq(0).sum()
            for columna in columnas_incertidumbre
        ],
        "valores_menores_a_0_001": [
            df[columna].lt(0.001).sum()
            for columna in columnas_incertidumbre
        ],
    }
)

display(
    resumen_minimos.style.format(
        {
            "minimo_exacto": "{:.10f}"
        }
    )
)

casos_vgeo_sigma_minima = (
    df.nsmallest(
        10,
        "Vgeo_sigma",
    )[
        [
            "Unique trajectory (identifier)",
            "IAU (code)",
            "Vgeo (km/s)",
            "Vgeo_sigma",
            "RAgeo_sigma",
            "DECgeo_sigma",
            "Qc (deg)",
            "MedianFitErr (arcsec)",
            "Num (stat)",
        ]
    ]
)

print("Diez trayectorias con menor incertidumbre de velocidad: ")
display(casos_vgeo_sigma_minima)


In [ ]:
# Marcar las trayectorias cuya incertidumbre de velocidad es exactamente cero

df["flag_vgeo_sigma_cero"] = (df["Vgeo_sigma"].eq(0))

cantidad_vgeo_sigma_cero = (df["flag_vgeo_sigma_cero"].sum())

porcentaje_vgeo_sigma_cero = 100 * cantidad_vgeo_sigma_cero / len(df)

print("Trayectorias con Vgeo_sigma igual a cero: ", cantidad_vgeo_sigma_cero)

print("Porcentaje de la base: ", round(porcentaje_vgeo_sigma_cero, 4), "%")

print("\n Conteo de la bandera: ")

display(
    df["flag_vgeo_sigma_cero"]
    .value_counts(dropna = False)
    .rename_axis("flag_vgeo_sigma_cero")
    .reset_index(name = "cantidad")
)


In [ ]:
# Visualizar las distribuciones centrales de los indicadores de calidad de las trayectorias

variables_continuas_calidad = {
    "RAgeo_sigma": "Incertidumbre de RA (grados)",
    "DECgeo_sigma": "Incertidumbre de DEC (grados)",
    "Vgeo_sigma": "Incertidumbre de velocidad (km/s)",
    "Qc (deg)": "Ángulo de convergencia Qc (grados)",
    "MedianFitErr (arcsec)": "Error mediano del ajuste (arcsec)",
}

figura, ejes = plt.subplots(
    nrows = 2,
    ncols = 3,
    figsize = (16, 9),
)

ejes = ejes.flatten()

for eje, (columna, etiqueta) in zip(ejes, variables_continuas_calidad.items()):
    percentil_99 = df[columna].quantile(0.99)
    datos_centrales = df.loc[df[columna] <= percentil_99, columna]
    mediana = df[columna].median()
    percentil_95 = df[columna].quantile(0.95)
    
    eje.hist(
        datos_centrales,
        bins = 40,
        color = "steelblue",
        edgecolor = "white",
        alpha = 0.85,
    )
    
    eje.axvline(
        mediana,
        color = "darkred",
        linestyle = "--",
        linewidth = 2,
        label = f"Mediana = {mediana:.3f}",
    )
    
    eje.axvline(
        percentil_95,
        color = "darkorange",
        linestyle = ":",
        linewidth = 2,
        label = f"P95 = {percentil_95:.3f}",
    )
    
    eje.set_title(f"{etiqueta} \n Se muestra el 99% central")
    eje.set_xlabel(etiqueta)
    eje.set_ylabel("Número de trayectorias")
    eje.legend()
    
conteo_estaciones = (df["Num (stat)"].value_counts().sort_index())

ejes[5].bar(
    conteo_estaciones.index.astype(str),
    conteo_estaciones.values,
    color = "slateblue",
    edgecolor = "white",
)

ejes[5].set_title("Número de estaciones participantes")
ejes[5].set_xlabel("Número de estaciones")
ejes[5].set_ylabel("Número de trayectorias")

plt.tight_layout()
plt.show()


In [ ]:
# Comparar las medianas de los indicadores de calidad según el ángulo Qc y el número de estaciones

intervalos_qc = [
    0,
    10,
    15,
    30,
    45,
    60,
    75,
    90,
]

etiquetas_qc = [
    "0–10",
    "10–15",
    "15–30",
    "30–45",
    "45–60",
    "60–75",
    "75–90",
]

df["grupo_qc"] = pd.cut(
    df["Qc (deg)"],
    bins = intervalos_qc,
    labels = etiquetas_qc,
    include_lowest = True,
    right = True,
)

resumen_por_qc = (
    df.groupby("grupo_qc", observed = False)
    .agg(
        trayectorias = (
            "Unique trajectory (identifier)", 
            "size"
        ),
        ra_sigma_mediana = (
            "RAgeo_sigma",
            "median",
        ),
        dec_sigma_mediana = (
            "DECgeo_sigma",
            "median",
        ),
        vgeo_sigma_mediana = (
            "Vgeo_sigma",
            "median",
        ),
        error_ajuste_mediano = (
            "MedianFitErr (arcsec)",
            "median",
        ),
        estaciones_mediana = (
            "Num (stat)", 
            "median",
        ),
    )
    .reset_index()
    .round(4)
)

print("Calidad según el ángulo de convergencia: ")
display(resumen_por_qc)

grupo_estaciones = (
    df["Num (stat)"]
    .astype("string")
    .mask(df["Num (stat)"] >= 7, "7+")
)

orden_estaciones = [
    "2",
    "3",
    "4",
    "5",
    "6",
    "7+",
]

df["grupo_estaciones"] = pd.Categorical(
    grupo_estaciones,
    categories = orden_estaciones,
    ordered = True,
)

resumen_por_estaciones = (
    df.groupby(
        "grupo_estaciones",
        observed = False,
    )
    .agg(
        
        trayectorias=(
            "Unique trajectory (identifier)",
            "size",
        ),
        qc_mediana=(
            "Qc (deg)",
            "median",
        ),
        ra_sigma_mediana=(
            "RAgeo_sigma",
            "median",
        ),
        dec_sigma_mediana=(
            "DECgeo_sigma",
            "median",
        ),
        vgeo_sigma_mediana=(
            "Vgeo_sigma",
            "median",
        ),
        error_ajuste_mediano=(
            "MedianFitErr (arcsec)",
            "median",
        ),
    )
    .reset_index()
    .round(4)
)

print("\nCalidad según el número de estaciones: ")
display(resumen_por_estaciones)


In [ ]:
# Comparar la precisión por número de estaciones dentro de intervalos semejantes de Qc

df["grupo_qc_reducido"] = pd.cut(
    df["Qc (deg)"],
    bins = [
        0,
        15,
        30,
        60,
        90,
    ],
    labels = [
        "0–15",
        "15–30",
        "30–60",
        "60–90",
    ],
    include_lowest = True,
    right = True,
)

grupo_estaciones_reducido = (
    df["Num (stat)"]
    .astype("string")
    .mask(
        df["Num (stat)"] >= 4,
        "4+",
    )
)

df["grupo_estaciones_reducido"] = pd.Categorical(
    grupo_estaciones_reducido,
    categories = [
        "2",
        "3",
        "4+",
    ],
    ordered = True,
)

resumen_combinado = (
    df.groupby(
        [
            "grupo_qc_reducido",
            "grupo_estaciones_reducido",
        ],
        observed = False,
    )
    .agg(
        trayectorias=(
            "Unique trajectory (identifier)",
            "size",
        ),
        ra_sigma_mediana=(
            "RAgeo_sigma",
            "median",
        ),
        dec_sigma_mediana=(
            "DECgeo_sigma",
            "median",
        ),
        vgeo_sigma_mediana=(
            "Vgeo_sigma",
            "median",
        ),
        error_ajuste_mediano=(
            "MedianFitErr (arcsec)",
            "median",
        ),
    )
    .reset_index()
    .round(4)
)

display(resumen_combinado)


In [ ]:
# Calcular una incertidumbre angular aproximada del radiante y auditar Vavg_sigma


declinacion_radianes = np.deg2rad(
    df["DECgeo (deg)"]
)

df["RAgeo_sigma_cielo"] = (
    df["RAgeo_sigma"]
    * np.cos(declinacion_radianes)
)

df["sigma_radiante_aprox"] = np.hypot(
    df["RAgeo_sigma_cielo"],
    df["DECgeo_sigma"],
)

df["flag_vavg_sigma_cero"] = (
    df["Vavg_sigma"].eq(0)
)

columnas_nuevas_calidad = [
    "RAgeo_sigma_cielo",
    "sigma_radiante_aprox",
    "Vavg_sigma",
]

resumen_nuevas_variables = (
    df[columnas_nuevas_calidad]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
    .T
    .rename(
        columns={
            "count": "cantidad",
            "mean": "media",
            "std": "desviacion_estandar",
            "min": "minimo",
            "1%": "percentil_1",
            "5%": "percentil_5",
            "25%": "percentil_25",
            "50%": "mediana",
            "75%": "percentil_75",
            "95%": "percentil_95",
            "99%": "percentil_99",
            "max": "maximo",
        }
    )
    .round(4)
)

display(resumen_nuevas_variables)

cantidad_vavg_sigma_cero = (
    df["flag_vavg_sigma_cero"].sum()
)

porcentaje_vavg_sigma_cero = (
    100
    * cantidad_vavg_sigma_cero
    / len(df)
)

print(
    "Trayectorias con Vavg_sigma igual a cero:",
    cantidad_vavg_sigma_cero,
)

print(
    "Porcentaje:",
    round(
        porcentaje_vavg_sigma_cero,
        4,
    ),
    "%",
)


## 5. Criterios de calidad


In [ ]:
# Construir muestras provisionales de calidad y cuantificar cuántas trayectorias y Gemínidas conserva cada una

df["cumple_qc_15"] = (
    df["Qc (deg)"].ge(15)
)

df["cumple_error_ajuste_60"] = (
    df["MedianFitErr (arcsec)"].le(60)
)

df["cumple_vavg_sigma_1"] = (
    df["Vavg_sigma"].gt(0)
    & df["Vavg_sigma"].le(1)
)

df["cumple_sigma_radiante_025"] = (
    df["sigma_radiante_aprox"].le(0.25)
)

df["calidad_radiante_publicada"] = (
    df["cumple_qc_15"]
    & df["cumple_error_ajuste_60"]
    & df["cumple_vavg_sigma_1"]
    & df["cumple_sigma_radiante_025"]
)

df["fov_completo"] = (
    df["Beg in (FOV)"]
    & df["End in (FOV)"]
)

df["calidad_estandar_provisional"] = (
    df["calidad_radiante_publicada"]
    & df["fov_completo"]
)

df["calidad_estricta_provisional"] = (
    df["calidad_estandar_provisional"]
    & df["Num (stat)"].ge(4)
)

criterios_individuales = {
    "Qc ≥ 15 grados": "cumple_qc_15",
    "Error de ajuste ≤ 60 arcsec": "cumple_error_ajuste_60",
    "0 < Vavg_sigma ≤ 1 km/s": "cumple_vavg_sigma_1",
    "Sigma del radiante ≤ 0.25 grados": "cumple_sigma_radiante_025",
    "Comienzo y final dentro del FOV": "fov_completo",
    "Cuatro o más estaciones": "Num (stat)",
}

resultados_criterios = []

for descripcion, columna in criterios_individuales.items():
    if columna == "Num (stat)":
        mascara = df[columna].ge(4)
    else:
        mascara = df[columna]
        
    cantidad = mascara.sum()
    
    resultados_criterios.append(
        {
            "criterio": descripcion,
            "cumplen": cantidad,
            "no_cumplen": len(df) - cantidad,
            "porcentaje_que_cumple": round(
                100 * cantidad / len(df),
                2,
            ),
        }
    )
    
resumen_criterios = pd.DataFrame(
        resultados_criterios
)
    
print("Retención de cada criterio por separado: ")
display(resumen_criterios)

es_geminida = df["IAU (code)"].eq("GEM")

muestras_provisionales = {
    "Base completa": pd.Series(
        True,
        index = df.index,
    ),
    "Calidad radiante publicada": (
        df["calidad_radiante_publicada"]
    ),
    "Estándar provisional": (
        df["calidad_estandar_provisional"]
    ),
    "Estricta provisional": (
        df["calidad_estricta_provisional"]
    ),
}

resultados_muestras = []

for nombre, mascara in muestras_provisionales.items():
    cantidad_total = mascara.sum()
    cantidad_geminidas = (
        mascara
        & es_geminida
    ).sum()
    
    resultados_muestras.append(
        {
            "muestra": nombre,
            "trayectorias_totales": cantidad_total,
            "porcentaje_de_la_base": round(
                100 * cantidad_total / len(df),
                2,
            ),
            "geminidas": cantidad_geminidas,
        }
    )
    
resumen_muestras = pd.DataFrame(
    resultados_muestras
)

print("\nRetención de las muestras combinadas: ")
display(resumen_muestras)


## 6. Relación entre calidad y brillo


In [ ]:
# Evaluar si los filtros de calidad retienen de forma diferente Gemínidas brillantes y débiles

geminidas_2019 = (
    df.loc[
        df["IAU (code)"].eq("GEM")
    ]
    .copy()
)

bordes_magnitud = np.arange(-5, 6, 1)

geminidas_2019["intervalo_magnitud"] = pd.cut(
    geminidas_2019["Peak (AbsMag)"],
    bins=bordes_magnitud,
    include_lowest=True,
    right=False,
)

retencion_por_magnitud = (
    geminidas_2019.groupby(
        "intervalo_magnitud",
        observed = False,
    )
    .agg(
        geminidas_totales = (
            "Unique trajectory (identifier)",
            "size",
        ),
        geminidas_estandar = (
            "calidad_estandar_provisional",
            "sum",
        ),
        geminidas_estrictas = (
            "calidad_estricta_provisional",
            "sum",
        ),
    )
    .reset_index()
)

retencion_por_magnitud = (
    retencion_por_magnitud.loc[
        retencion_por_magnitud["geminidas_totales"] > 0
    ]
    .copy()
)

retencion_por_magnitud["retencion_estandar_porcentaje"] = (
    100
    * retencion_por_magnitud["geminidas_estandar"]
    / retencion_por_magnitud["geminidas_totales"]
)

retencion_por_magnitud["retencion_estricta_porcentaje"] = (
    100
    * retencion_por_magnitud["geminidas_estrictas"]
    / retencion_por_magnitud["geminidas_totales"]
)

retencion_por_magnitud = (retencion_por_magnitud.round(2))

print("Retención de Gemínidas según magnitud: ")
display(retencion_por_magnitud)

resumen_magnitud_muestras = pd.DataFrame(
    {
        "muestra": [
            "Gemínidas completas",
            "Gemínidas estándar",
            "Gemínidas estrictas",
        ],
        "cantidad": [
            len(geminidas_2019),
            geminidas_2019[
                "calidad_estandar_provisional"
            ].sum(),
            geminidas_2019[
                "calidad_estricta_provisional"
            ].sum(),
        ],
        "magnitud_media": [
            geminidas_2019[
                "Peak (AbsMag)"
            ].mean(),
            geminidas_2019.loc[
                geminidas_2019[
                    "calidad_estandar_provisional"
                ],
                "Peak (AbsMag)",
            ].mean(),
            geminidas_2019.loc[
                geminidas_2019[
                    "calidad_estricta_provisional"
                ],
                "Peak (AbsMag)",
            ].mean(),
        ],
        "magnitud_mediana": [
            geminidas_2019[
                "Peak (AbsMag)"
            ].median(),
            geminidas_2019.loc[
                geminidas_2019[
                    "calidad_estandar_provisional"
                ],
                "Peak (AbsMag)",
            ].median(),
            geminidas_2019.loc[
                geminidas_2019[
                    "calidad_estricta_provisional"
                ],
                "Peak (AbsMag)",
            ].median(),
        ],
    }
).round(3)

print("\nComparación general de magnitud: ")
display(resumen_magnitud_muestras)


In [ ]:
# Identificar qué criterios de calidad dependen del brillo y comparar la calidad residual de la muestra estándar

geminidas_2019["grupo_brillo"] = pd.cut(
    geminidas_2019["Peak (AbsMag)"],
    bins = [-np.inf, -1, 0, 1, np.inf],
    labels = ["M ≤ -1", "-1 < M ≤ 0", "0 < M ≤ 1", "M > 1"],
    include_lowest = True,
    right = True
)

diagnostico_filtros_brillo = (
    geminidas_2019.groupby("grupo_brillo", observed = False)
    .agg(
        geminidas_totales = (
            "Unique trajectory (identifier)", 
            "size",
        ),
        cumple_qc_15 = (
            "cumple_qc_15",
            "mean",
        ),
        cumple_error_ajuste_60=(
            "cumple_error_ajuste_60",
            "mean",
        ),
        cumple_vavg_sigma_1=(
            "cumple_vavg_sigma_1",
            "mean",
        ),
        cumple_sigma_radiante_025=(
            "cumple_sigma_radiante_025",
            "mean",
        ),
        cumple_fov_completo=(
            "fov_completo",
            "mean",
        ),
        cumple_estandar=(
            "calidad_estandar_provisional",
            "mean",
        ),
        cumple_estricta=(
            "calidad_estricta_provisional",
            "mean",
        ),
    )
    .reset_index()
)

columnas_porcentaje = [
    "cumple_qc_15",
    "cumple_error_ajuste_60",
    "cumple_vavg_sigma_1",
    "cumple_sigma_radiante_025",
    "cumple_fov_completo",
    "cumple_estandar",
    "cumple_estricta",
]

diagnostico_filtros_brillo[columnas_porcentaje] = 100 * diagnostico_filtros_brillo[columnas_porcentaje]

diagnostico_filtros_brillo = diagnostico_filtros_brillo.round(2)

print("Porcentaje que supera cada criterio según brillo: ")
display(diagnostico_filtros_brillo)

geminidas_estandar_2019 = (
    geminidas_2019.loc[
        geminidas_2019[
            "calidad_estandar_provisional"
        ]
    ]
    .copy()
)

calidad_residual_por_brillo = (
    geminidas_estandar_2019.groupby(
        "grupo_brillo",
        observed=False,
    )
    .agg(
        geminidas=(
            "Unique trajectory (identifier)",
            "size",
        ),
        sigma_radiante_mediana=(
            "sigma_radiante_aprox",
            "median",
        ),
        qc_mediana=(
            "Qc (deg)",
            "median",
        ),
        vavg_sigma_mediana=(
            "Vavg_sigma",
            "median",
        ),
        error_ajuste_mediano=(
            "MedianFitErr (arcsec)",
            "median",
        ),
        estaciones_mediana=(
            "Num (stat)",
            "median",
        ),
    )
    .reset_index()
    .round(4)
)

print("\n Calidad restante dentro de la muestra estándar: ")

display(calidad_residual_por_brillo)

